# シナリオ:サポート対応時間の長期化要因分析

## 1. データセット構成

### テーブルデータ（CSV・5ファイル）

| ファイル | 役割 | 主なカラム | 件数 |
|---|---|---|---|
| support_tickets.csv | ファクト（チケット） | ticket_id, customer_id, product_id, agent_id, category, priority, channel, open_date, close_date, resolution_time_hours, reopened_count, csat_score, status | 60件 |
| customers.csv | ディメンション（顧客） | customer_id, customer_name, segment, region, industry, plan_tier, mrr_jpy, contract_start_date | 30社 |
| agents.csv | ディメンション（担当者） | agent_id, agent_name, team, tier, location, hire_date | 10名 |
| products.csv | ディメンション（製品） | product_id, product_name, module, release_version | 5製品 |
| sla_policies.csv | リファレンス（SLA基準） | priority, sla_target_hours, business_hours_only | 4行 |

- カテゴリは6種類（データ同期エラー／ログイン認証エラー／決済・請求エラー／パフォーマンス低下／UI表示崩れ／機能に関する質問）
- 「データ同期エラー」だけが平均解決時間・再発率・SLA違反率・CSATのいずれも著しく悪いことは確認可能なものの、テーブルだけでは「何が悪いか」は分かるが「なぜ」は分からない状態

### ボリュームファイル（非構造化・4ファイル）

いずれも「データ同期エラー」の中心ケース（TICK-10204 ほか、顧客 さくらリテール株式会社／CUST-0007）に関する文書で、ここに根本原因（外部連携APIプロバイダ Connector社による無通知のレート制限変更）が記述されています。

- `escalation_email_TICK-10204.pdf`：顧客からのエスカレーションメール
- `tier2_handoff_notes_TICK-10204.docx`：Tier1→Tier2 引継ぎ調査メモ
- `data_sync_incident_review.pptx`：社内インシデントレビュー資料（RCA）
- `error_screenshot_TICK-10204.jpg`：顧客環境のエラー画面スクリーンショット

## 2. 想定QA（分析軸ごと）

各軸について、**Step1：テーブルのみで分かること**と、**Step2：ボリューム追加で分かること**を整理

### 軸1：カテゴリ別の解決時間

* **Step1（テーブルのみ）**：「カテゴリ別の平均解決時間を教えて」<br>
  → 回答例：データ同期エラーが平均61.6時間で最長。次いでパフォーマンス低下12.8時間、決済/請求エラー14.1時間、ログイン認証エラー9.7時間、UI表示崩れ5.2時間、機能に関する質問3.1時間。
* **Step2（ボリューム追加）**：「データ同期エラーの解決時間が他カテゴリより突出して長い理由は？」<br>
  → PDF/DOCXの記述から、外部連携先 Connector社が事前通知なくAPIレート制限を変更したため、CloudBridge側の同期バッチが対応できず再発を繰り返していたことが根本原因と分かる。

### 軸2：SLA遵守状況（優先度別）

* **Step1（テーブルのみ）**：「優先度ごとのSLA違反率が最も高いカテゴリはどこ？」<br>
  → 回答例：データ同期エラーはSLA違反率100%（10件中10件がSLA目標を超過）。パフォーマンス低下50%、決済/請求エラー40%、ログイン認証エラー20%、UI表示崩れ10%、機能に関する質問0%。
* **Step2（ボリューム追加）**：「データ同期エラーでSLAを守れなかった具体的な経緯は？」<br>
  → DOCXの調査タイムラインから、Tier1での通常対応では解決できず、Tier2・エンジニアリングへのエスカレーションと外部要因の調査に時間を要したことがSLA超過の直接的な経緯として分かる。

### 軸3：顧客満足度（CSAT）との関係

* **Step1（テーブルのみ）**：「カテゴリ別の平均CSATスコアを教えて」<br>
  → 回答例：データ同期エラーは平均1.4（5点満点）で最も低い。他カテゴリは4.1〜4.7と高水準。
* **Step2（ボリューム追加）**：「CSATが低いデータ同期エラーのチケットで、顧客は具体的に何に不満を述べているか？」<br>
  → PDFのエスカレーションメール内の顧客コメント（「同じ問題が3回も繰り返されており…契約更新の判断を保留せざるを得ない」）から、再発そのものと恒久対応の欠如への不満が低CSATの直接要因であることが分かる。

## 3. まとめ

テーブルのみでは一貫して「何が悪いか（What）」までしか分からず、ボリューム追加後に初めて「なぜ悪いのか（Why）」＝Connector社のAPIレート制限変更という共通の根本原因に到達可能
